<div class="alert alert-success"><h1>Urban Heat Island prediction and classification</h1></div>

### Extracting data from GEE

In [1]:
import geemap
import ee

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project = "ee-leviekytz")

In [ ]:
Map = geemap.Map()

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Define the region of interest
kenya = ee.FeatureCollection("projects/ee-leviekytz/assets/ken_admbnda_adm2_iebc_20191031")
mvita = kenya.filter(ee.Filter.eq('ADM2_EN','Mvita'))
roi = mvita.geometry()
Map.centerObject(roi, 15)

#A function to filter out cloudy pixels
def cloudMask(cloudyScene):
    scored = ee.Algorithms.Landsat.simpleCloudScore(cloudyScene)
    mask = scored.select(['cloud']).lte(10)
    return cloudyScene.updateMask(mask)

#Generate a water mask
water = ee.Image('JRC/GSW1_0/GlobalSurfaceWater').select('occurrence')
notWater = water.mask().Not()

In [ ]:

#Landsat collection filtering
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
filtered = (landsat.filter(ee.Filter.lt('CLOUD_COVER',20))
.filterDate('2014-01-01','2026-01-01')
.filterBounds(roi)
.map(cloudMask))

#Finding the composite median and clipping the landsat image
landsat_clean = filtered.median().clip(roi)

#Visualizing the landsat image
rgbVis = {
    'bands': ['B4','B3', 'B2'],
    'min':0.0,
    'max':0.4
}
Map.addLayer(landsat_clean, rgbVis,"Mvita_landsat")

Map(bottom=4289085.0, center=[-4.051297205968075, 39.66242191294893], controls=(WidgetControl(options=['positi…

In [38]:
#Getting information about the landsat image
print('Landsat collection', landsat_clean.getInfo())

#Select the thermal band 10
thermal = (landsat_clean.select('B10')
.clip(roi)
.updateMask(notWater))

Map.addLayer(thermal,{
    'min':295,
    'max':310,
    'palette': ['blue','white','red']
},"Landsat_BT")
Map

Landsat collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'origin': [39, -5], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'origin': [39, -5], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'origin': [39, -5], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'origin': [39, -5], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'origin': [39, -5], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [2, 2], 'orig

Map(bottom=2144757.0, center=[-4.056826751206199, 39.67334747314454], controls=(WidgetControl(options=['positi…